In [34]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(1, '../../scripts/')
from utils.load_environmental_variables import *
prebuild = '/data2/hratch/human_me/prebuild/'

In [92]:
am = pd.read_excel(prebuild + 'Gregersen_mrna_turnover_unprocessed.xlsx')

# formatting
am.set_index('Name', inplace = True)
am = am.iloc[:,:2] # only controls replicates
am = np.log(2)/(am/60) # convert to turnover rate (hrs^-1) 
am = pd.DataFrame({'median_turnover': am.median(axis = 1)})
print('The median mrna turnover before mapping ids or further manipulation is {:.4f}'.format(am.median()[0]))

The median mrna turnover before mapping ids or further manipulation is 0.0611


In [94]:
# mapping

psim_me = pd.read_hdf('/data2/hratch/human_me/temp_psim.h5', key = 'coupling_params') # can use final psim me as well, all ids are the same
am_ids = pd.DataFrame(data = {'GENE_SYMBOL': am.index})

# first conserve existing mapping in PSIM
temp = psim_me[psim_me.GENE_SYMBOL.isin(am_ids.GENE_SYMBOL)][['GENE_SYMBOL', 'HGNC_ID']]
cond = temp.GENE_SYMBOL.unique().shape[0] == temp.HGNC_ID.unique().shape[0] == temp.shape[0]
if not cond:
    raise ValueError('Was working with 1-to-1 mapping; if no longer, see protein_turnover script')
mapper = dict(zip(temp.GENE_SYMBOL, temp.HGNC_ID))
am_ids['HGNC_ID'] = am_ids.GENE_SYMBOL.map(mapper)

# try mapping additional missing IDs
p1 = am_ids[am_ids.HGNC_ID.notna()]
p2 = am_ids[am_ids.HGNC_ID.isna()]

ehm = pd.read_csv(prebuild + 'sequence_information/identifiers.txt', sep = '\t')
ehm = ehm[ehm['Approved symbol'].notna() & ehm['HGNC ID'].notna() & ehm['Approved symbol'].isin(p2.GENE_SYMBOL)]
ehm = ehm[['HGNC ID', 'Approved symbol']]
ehm.drop_duplicates(inplace = True)

ehm.drop_duplicates(inplace = True)
cond = ehm['HGNC ID'].unique().shape[0] == ehm['Approved symbol'].unique().shape[0] == ehm.shape[0]
if not cond:
    raise ValueError('Expected 1 to 1 mapping, consider expanding for redundant HGNCs (see protein_turnover script)')
p2['HGNC_ID'] = p2.GENE_SYMBOL.map(dict(zip(ehm['Approved symbol'], ehm['HGNC ID'])))

In [102]:
am_ids = pd.concat([p1, p2], axis = 0, ignore_index=True)
am['HGNC_ID'] = am.index.map(dict(zip(am_ids.GENE_SYMBOL, am_ids.HGNC_ID)))
am['GENE_SYMBOL'] = am.index
am.reset_index(inplace = True, drop = True)
am = pd.concat([am.iloc[:,-2:], am.iloc[:,:-2]], axis = 1)
am.to_csv(build_files_path + 'Gregersen_mrna_turnover_processed.tsv', sep = '\t')
